# Flow compensation on a spiral, without a spiral kernel

SeqCraft ships a packaged kernel for the Cartesian gradient echo, and
[`gre_2d/03_flow_comp`](../gre_2d/03_flow_comp.ipynb) asks it for flow compensation with one
keyword. There is no packaged spiral kernel — [`01_build`](01_build.ipynb) composes a spiral
repetition out of `Excitation`, `SpiralReadout` and a spoiler, in ordinary notebook code.

This notebook adds flow compensation to that composition. **No kernel class is written**, and the
physics is designed by the same machinery the packaged kernel uses.

## The problem

A spiral-in trajectory starts at the edge of k-space and ends at the origin, so the arm plays in
full **before** the echo and arrives carrying both moments:

$$\phi = 2\pi\left(m_0 x_0 + m_1 v\right)$$

The prephaser that takes `k` out to the edge has to cancel $m_0$ at the echo — that is what puts
$k = 0$ there. Nothing makes it cancel $m_1$, so a spin moving in the slice plane arrives with a
velocity-dependent phase, exactly as on a Cartesian readout.

What makes this worth showing is the **state**. Each interleaf is the same trajectory rotated, so
the moment the prephaser must cancel rotates with it, on both in-plane axes at once:

```text
angle     m0_x      m0_y          the arm alone, measured to its own echo
0.000  -133.33      0.00
0.785   -94.28    -94.28
1.571    -0.00   -133.33
```

A Cartesian phase encode varies one axis linearly and leaves the other alone. This varies both,
and no packaged kernel in SeqCraft produces it.

| | |
|---|---|
| **1** | the composition, and what it hands over |
| **2** | declaring the region the designer may own |
| **3** | designing once for the family |
| **4** | measuring the emitted repetitions |
| **5** | one echo time, and a longer one |
| **6** | the file |

**Needs nothing but `seqcraft`.**

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pypulseq as pp

import seqcraft as sc

opts = pp.Opts(
    max_grad=38, grad_unit='mT/m',
    max_slew=140, slew_unit='T/m/s',
    rf_dead_time=100e-6,
    rf_ringdown_time=30e-6,
    adc_dead_time=10e-6,
    adc_samples_limit=8192,
)

FOV_MM, MATRIX, THICKNESS_MM = 240.0, 64, 5.0
SHOTS, FLIP_DEG, TR_S = 8, 15.0, 30e-3

SEQ_DIR = Path('seq')
SEQ_DIR.mkdir(exist_ok=True)

/opt/homebrew/Caskroom/miniforge/base/envs/seqcraft-dev/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

## 1. The composition, and what it hands over

The same three leaves `01_build` uses, with two changes.

`variant='in'` puts the echo at the end of the arm rather than 12 us into it, which is what gives
the first moment something to be. And two components are asked to **not realise** a region they
would normally own:

```text
arm(..., prephase=False)     the prephaser that takes k to the edge
exc(rephase=False)           the slice rephaser
```

Both already support that — it is how `CartesianLine(prephase=False)` and
`Excitation.build(rephase=False)` let a caller fold those waveforms into something larger. Here
the something larger is one region designed for all three axes at once.

In [2]:
exc = sc.modules.Excitation(opts=opts, flip_deg=FLIP_DEG, thickness_mm=THICKNESS_MM,
                            duration_s=1e-3)
arm = sc.modules.SpiralReadout(opts=opts, fov_mm=FOV_MM, matrix=MATRIX, shots=SHOTS,
                               dwell_s=4e-6, variant='in')
spoil = sc.modules.spoiler(opts, cycles_per_voxel=4.0, voxel_mm=THICKNESS_MM, axis='z')

angles = tuple(2.0 * np.pi * i / SHOTS for i in range(SHOTS))

print(f'arm            {arm.duration_s * 1e3:.3f} ms, starting k {arm.start_k_per_m:.1f} 1/m')
print(f'its prephaser  {arm.prephaser_duration_s * 1e6:.0f} us, which this notebook takes over')
print(f'its echo       {arm.time_to_echo(0) * 1e6:.0f} us into a block that INCLUDES that '
      f'prephaser')
print(f'interleaves    {SHOTS}')

arm            2.770 ms, starting k 133.3 1/m
its prephaser  300 us, which this notebook takes over
its echo       3044 us into a block that INCLUDES that prephaser
interleaves    8


---

## 2. Declaring the region the designer may own

`sc.PhysicalDesignScope` says where the spins were excited, what plays on either side of the
region, which axes may be used inside it, and which states the repetition comes in:

```text
0            origin_s                  window                     echo
|               |                        |                          |
[---- before ---X------------[==== designed here ====]---- after ---X
```

**One subtlety, and it is the one that bites.** `arm.time_to_echo(0)` is measured from a block
that includes the arm's *own* prephaser. This scope has taken that prephaser over, so its duration
has to come back off — otherwise the declared echo lands past the end of the arm, inside the
rewinder, and the design is solved for an instant the sequence does not have. It does not raise;
the moments simply come out non-zero.

The rule behind it: **what you declare must describe the configuration you are emitting.**

`design_states` is a hint, not a promise. The arm is one trajectory rotated, so on `x` the
requirement is a state-independent magnitude times `cos(angle)` and on `y` times `sin(angle)` —
both largest at the axis-aligned angles. Those two are worth designing against. If that reasoning
were wrong, SeqCraft would find out: it realises **every** interleaf before accepting a schedule.

In [3]:
scope = sc.PhysicalDesignScope(
    origin_s=exc.time_to_center(),
    before=exc(rephase=False),
    after=lambda angle: arm(angle_rad=angle, prephase=False),
    echo_in_after_s=arm.time_to_echo(0) - arm.prephaser_duration_s,
    axes=('x', 'y', 'z'),
    states=angles,
    design_states=(angles[0], angles[SHOTS // 4]),
    min_window_s=arm.prephaser_duration_s,
)

print(f'axes owned      {scope.axes}')
print(f'states          {len(scope.states)} interleaf angles')
print(f'designing from  ' + ', '.join(f'{a:.3f} rad' for a in scope.design_states))

axes owned      ('x', 'y', 'z')
states          8 interleaf angles
designing from  0.000 rad, 1.571 rad


---

## 3. Designing once for the family

The intent is the same object a packaged kernel takes. It says *what physical relation must
hold*; where that is realised is the scope's business, and which waveform realises it is
SeqCraft's.

In [4]:
design = sc.design_repetition(scope, opts=opts,
                              flow_comp=sc.FlowCompensation(axis=('x', 'y', 'z')))

print(f'window   {design.window_s * 1e6:.0f} us')
print(f'TE       {design.te_s * 1e3:.3f} ms, the same for every interleaf')

window   1450 us
TE       4.834 ms, the same for every interleaf


A repetition is built **once**, from the finished design. Nothing goes back and edits a block that
already exists — `design.repetition(angle)` returns the excitation, the designed region and the
arm as one block, and the spoiler is added the way `01_build` adds it.

In [5]:
def repetition_of(at, angle, *, acquire=True):
    """One interleaf of a design: the designed repetition, plus the spoiler."""
    out = at.repetition(angle)
    out.add(out.duration, spoil)
    return out


def repetition(angle, **kw):
    return repetition_of(design, angle, **kw)


print(f'one repetition: {repetition(angles[0]).duration * 1e3:.3f} ms, in TR {TR_S * 1e3:.0f} ms')

one repetition: 6.290 ms, in TR 30 ms


---

## 4. Measuring the emitted repetitions

Every interleaf, all three axes, both moments, integrated over everything the repetition plays
between the excitation and the echo.

Measured **from the excitation instant** rather than from the start of the block: on `x` and `y`
the two agree, but half the slice-selection lobe plays before the RF centre and dephases nothing,
because there is no transverse magnetisation yet.

In [6]:
def moment(at, angle, order, axis):
    """
    One gradient moment of a whole emitted repetition, excitation to echo.

    Ordinary measurement code over the emitted events: integrate every gradient on the axis
    between the two instants, clipping the lobes that straddle either end. None of this is part
    of declaring the design — it is how we check what came out.
    """
    origin, echo = scope.origin_s, scope.origin_s + at.te_s
    total = 0.0
    for start, event, _ in sc.flatten(repetition_of(at, angle)):
        if getattr(event, 'channel', None) != axis:
            continue
        times, amps = sc.events.knots_of(event, start)
        if times.size < 2 or echo <= times[0] or times[-1] <= origin:
            continue
        if times[0] < origin:                      # a lobe straddling the excitation instant
            cut = int(np.searchsorted(times, origin))
            edge = float(np.interp(origin, times, amps))
            times = np.concatenate(([origin], times[cut:]))
            amps = np.concatenate(([edge], amps[cut:]))
        if echo < times[-1]:                       # and one straddling the echo
            cut = int(np.searchsorted(times, echo))
            edge = float(np.interp(echo, times, amps))
            times = np.concatenate((times[:cut], [echo]))
            amps = np.concatenate((amps[:cut], [edge]))
        total += sc.events.pwl_moment(times - origin, amps, order)
    return float(total)


print(f'{"angle":>8}{"m0_x":>12}{"m0_y":>12}{"m0_z":>12}'
      f'{"m1_x":>12}{"m1_y":>12}{"m1_z":>12}')
for angle in angles:
    print(f'{angle:8.3f}' + ''.join(f'{moment(design, angle, o, a):12.2e}'
                                    for o in (0, 1) for a in ('x', 'y', 'z')))

   angle        m0_x        m0_y        m0_z        m1_x        m1_y        m1_z
   0.000    0.00e+00    3.55e-15   -1.14e-13    5.55e-17   -2.78e-17   -2.22e-16
   0.785    2.84e-14    0.00e+00   -1.14e-13    2.78e-17   -2.78e-17   -2.22e-16
   1.571    0.00e+00    0.00e+00   -1.14e-13    1.39e-17    5.55e-17   -2.22e-16
   2.356    0.00e+00    2.84e-14   -1.14e-13    2.78e-17    2.78e-17   -2.22e-16
   3.142    0.00e+00    0.00e+00   -1.14e-13   -5.55e-17    1.39e-17   -2.22e-16
   3.927   -2.84e-14   -1.42e-14   -1.14e-13   -5.55e-17    0.00e+00   -2.22e-16
   4.712    3.55e-15    2.84e-14   -1.14e-13   -1.39e-17    0.00e+00   -2.22e-16
   5.498    1.42e-14   -2.84e-14   -1.14e-13    0.00e+00   -2.78e-17   -2.22e-16


`k` arrives at the origin on all three axes and the first moment is gone, for every interleaf.

The `z` column is worth a second look. The slice rephasing was handed over with
`exc(rephase=False)`, so the designed region is doing two jobs there at once: returning `k_z` to
zero, which the rephaser used to do on its own, and nulling the first moment, which it never did.
Those are one problem on that axis, and asking for only the second would have bought `m1 = 0` by
leaving `k_z` somewhere other than zero.

What it buys physically: a spin moving at constant velocity reaches the echo with the phase it
would have had standing still, whichever interleaf is playing. Without it, that phase rotates from
shot to shot with the trajectory — a shot-to-shot inconsistency in a reconstruction that assumes
every interleaf saw the same object.

**Image-level artefact reduction is not demonstrated here.** What is demonstrated is that the
velocity-dependent phase term goes to zero on the emitted waveform.

---

## 5. One echo time, and a longer one

The schedule is a property of the **family**, not of each state: solving it per interleaf would
give eight different echo times for one image. Designing it once is what makes them comparable.

And because the family can be designed again at any legal longer time, a protocol holding two
different repetition families can take the longer of their two minima and ask both for it — with
no acquisition-wide timing manager anywhere.

In [7]:
print(f'{"TE / ms":>10}{"window / us":>13}{"worst |m1| / (s/m)":>21}')
for extra_ms in (0.0, 0.5, 2.0):
    at = design if extra_ms == 0.0 else design.at(te_s=design.te_s + extra_ms * 1e-3)
    worst = max(abs(moment(at, angle, 1, axis))
                for angle in angles for axis in ('x', 'y', 'z'))
    print(f'{at.te_s * 1e3:10.3f}{at.window_s * 1e6:13.0f}{worst:21.2e}')

   TE / ms  window / us   worst |m1| / (s/m)
     4.834         1450             2.22e-16
     5.334         1950             2.22e-16
     6.834         3450             2.22e-16


---

## 6. The file

In [8]:
scan = sc.LogicBlock('gre_spiral_2d_flow_comp')
for index, angle in enumerate(angles):
    scan.add(index * TR_S, repetition(angle))

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', sc.SeqCraftWarning)
    seq = sc.compile(scan, opts, name='gre_spiral_2d_flow_comp')

path = SEQ_DIR / 'gre_spiral_2d_flow_comp.seq'
seq.write(str(path))
print(f'{len(seq.block_events)} blocks, {seq.duration()[0]:.3f} s   {path}')
for warning in caught:
    print(f'\n{str(warning.message).split(":")[0]}')

47 blocks, 0.216 s   seq/gre_spiral_2d_flow_comp.seq

24 same-axis gradient merges

16 blocks over the vector-norm limit (legal on real amplifiers)


---

## Summary

A spiral-in arm reaches the echo carrying a first moment, and the prephaser that puts $k = 0$
there does nothing about it. Handing that prephaser to SeqCraft — along with the slice rephaser —
lets one designed region null both moments on all three axes, for every interleaf, at one echo
time. Measured on the emitted repetitions, $m_0$ and $m_1$ are at the arithmetic floor.

The point of the exercise is where the design happened. This repetition has no kernel class; it is
`Excitation`, `SpiralReadout` and a spoiler composed in a notebook. `sc.PhysicalDesignScope` is
how such a composition says *this region is yours* — and what it reaches is the same designer
`GRE2DTR` uses for its own winder. **A composition does not have to become a packaged module to
get physical design.**

**What this notebook does not cover.** A scope owns one adjustable region, so a repetition wanting
its prephaser and its rewinder designed together does not fit. What plays between that region and
the echo must be the same duration for every state — true of a rotated arm, not of a
variable-length one. The designed region always returns `k` to the origin at the echo on the axes
it owns, so a composition wanting a non-zero `k` there, as a Cartesian phase encode does, is what
the packaged kernels are for. And only the zeroth and first moments are represented.

## References

Bernstein, King and Zhou, *Handbook of MRI Pulse Sequences*, Elsevier 2004 — §9.2 for gradient
moment nulling, §17.6 for spiral trajectories and their gradient design.

Nishimura, Irarrazabal and Meyer, *A velocity k-space analysis of flow effects in echo-planar and
spiral imaging*, Magn Reson Med 33(4):549-556, 1995 — why the moment structure of a spiral differs
from a Cartesian readout, and what it does to flowing spins.